Script to parse ACL Anthology bibtex. Results of the parsing saved in pickle format for faster loading.

In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore", module=r"acl_anthology\..*")

Takes approximately 6 minutes and 30 seconds to load the ACL Anthology data

In [2]:
from acl_anthology import Anthology

# Clones/updates the official ACL Anthology data repo (first run takes a few minutes)
anthology = Anthology.from_repo()

# Optional: we're iterating over everything, so loading all at once can be faster
anthology.load_all()

/home/tom/Hallucination-KG-Survey/venv/lib/python3.12/site-packages/acl_anthology/people/index.py:751: NameSpecResolutionWarning: More than one NameSpecification resolves to 'bing-liu/unverified' on the same item (('2020.coling', 'main', '463'))
  warnings.warn(NameSpecResolutionWarning(namespec, message))
/home/tom/Hallucination-KG-Survey/venv/lib/python3.12/site-packages/acl_anthology/people/index.py:751: NameSpecResolutionWarning: More than one NameSpecification resolves to 'zhiyuan-liu/unverified' on the same item (('2020.emnlp', 'main', '515'))
  warnings.warn(NameSpecResolutionWarning(namespec, message))
/home/tom/Hallucination-KG-Survey/venv/lib/python3.12/site-packages/acl_anthology/people/index.py:751: NameSpecResolutionWarning: More than one NameSpecification resolves to 'vaibhav-kumar/unverified' on the same item (('2020.nlposs', '1', '5'))
  warnings.warn(NameSpecResolutionWarning(namespec, message))
/home/tom/Hallucination-KG-Survey/venv/lib/python3.12/site-packages/acl_an

Anthology(datadir=PosixPath('/home/tom/.local/share/acl-anthology/git/acl-org-acl-anthology-git/data'), verbose=False)

In [3]:
START_YEAR = 2020
SEP = "  and"  # same separator as the old BibTeX-derived records

def name_bibtex(namespec):
    n = namespec.name
    return f"{n.last}, {n.first}" if n.first else n.last

def paper_to_record(paper):
    authors = list(paper.authors)
    return {
        "bib_key": paper.bibkey,
        "ENTRYTYPE": paper.bibtype,
        "title": str(paper.title),
        "abstract": str(paper.abstract) if paper.abstract else None,
        "author": SEP.join(name_bibtex(a) for a in authors),
        "booktitle": str(paper.parent.title) if paper.parent.title else None,
        "year": paper.year,
        "month": paper.month,
        "pages": paper.pages,
        "url": paper.web_url,
        "publisher": paper.publisher,
        "anthology_id": paper.full_id,
        "venues": list(paper.venue_ids),
        "doi": paper.doi,
        "language": paper.language,
    }

def iter_records(start_year):
    for volume in anthology.volumes():
        # year is inherited from the volume, so filter at volume level (fast)
        if not volume.year or int(volume.year) < start_year:
            continue
        for paper in volume.papers():
            # skip frontmatter and retracted/removed papers
            if paper.is_frontmatter or paper.is_deleted:
                continue
            yield paper_to_record(paper)

Takes approximately 1 minute and 30 seconds to load into a DataFrame.

In [4]:
# Build the DataFrame once from all records (never append row by row)
acl_bibliography_df = pd.DataFrame.from_records(iter_records(START_YEAR))

print(f"Extracted {len(acl_bibliography_df):,} papers from {START_YEAR} onwards")

/home/tom/Hallucination-KG-Survey/venv/lib/python3.12/site-packages/acl_anthology/text/texmath.py:211: TeXParserWarning: Unknown TeX-math command: \choose
  warnings.warn(TeXParserWarning(code, f"Unknown TeX-math command: {code}"))
/home/tom/Hallucination-KG-Survey/venv/lib/python3.12/site-packages/acl_anthology/text/texmath.py:211: TeXParserWarning: Unknown TeX-math command: \textcolor{red}{100~trillion}
  warnings.warn(TeXParserWarning(code, f"Unknown TeX-math command: {code}"))
/home/tom/Hallucination-KG-Survey/venv/lib/python3.12/site-packages/acl_anthology/text/texmath.py:211: TeXParserWarning: Unknown TeX-math command: \binom{62}{6}
  warnings.warn(TeXParserWarning(code, f"Unknown TeX-math command: {code}"))
/home/tom/Hallucination-KG-Survey/venv/lib/python3.12/site-packages/acl_anthology/text/texmath.py:211: TeXParserWarning: Unknown TeX-math command: \textcolor{red}{WARNING: This paper contains examples of harmful or inappropriate images generated by models.}
  warnings.warn(Te

Extracted 76,671 papers from 2020 onwards


In [5]:
# List number of NaN for columns that have missing values
for column in acl_bibliography_df.columns:
    num_missing = acl_bibliography_df[column].isna().sum()
    if num_missing > 0:
        print(f"{column}: {num_missing} missing values")

abstract: 4458 missing values
month: 810 missing values
pages: 128 missing values
publisher: 29 missing values
doi: 18249 missing values
language: 73975 missing values


In [6]:
# Print example
print(acl_bibliography_df.iloc[0].to_string())

bib_key                                      wang-etal-2020-touch
ENTRYTYPE                                           inproceedings
title           Touch Editing: A Flexible One-Time Interaction...
abstract        We propose a touch-based editing method for tr...
author          Wang, Qian  andZhang, Jiajun  andLiu, Lemao  a...
booktitle       Proceedings of the 1st Conference of the Asia-...
year                                                         2020
month                                                    December
pages                                                        1–11
url                    https://aclanthology.org/2020.aacl-main.1/
publisher               Association for Computational Linguistics
anthology_id                                     2020.aacl-main.1
venues                                                     [aacl]
doi                                  10.18653/v1/2020.aacl-main.1
language                                                      NaN


In [7]:
acl_bibliography_df.head(2)

,bib_key,ENTRYTYPE,title,abstract,author,booktitle,year,month,pages,url,publisher,anthology_id,venues,doi,language
0,wang-etal-2020-touch,inproceedings,Touch Editing: A Flexible One-Time Interaction...,We propose a touch-based editing method for tr...,"Wang, Qian andZhang, Jiajun andLiu, Lemao a...",Proceedings of the 1st Conference of the Asia-...,2020,December,1–11,https://aclanthology.org/2020.aacl-main.1/,Association for Computational Linguistics,2020.aacl-main.1,[aacl],10.18653/v1/2020.aacl-main.1,NaN
1,chi-etal-2020-monolingual,inproceedings,Can Monolingual Pretrained Models Help Cross-L...,Multilingual pretrained language models (such ...,"Chi, Zewen andDong, Li andWei, Furu andMao,...",Proceedings of the 1st Conference of the Asia-...,2020,December,12–17,https://aclanthology.org/2020.aacl-main.2/,Association for Computational Linguistics,2020.aacl-main.2,[aacl],10.18653/v1/2020.aacl-main.2,NaN


In [8]:
OUT = "../data/acl_anthology.parquet"

acl_bibliography_df.to_parquet(OUT, engine="pyarrow", compression="zstd", index=False)

# Reload check
loaded = pd.read_parquet(OUT)
print(loaded.shape)

(76671, 15)
